# 02 Champion Models Spatial Generalization

Tests champion-model transfer to geographically distinct regions.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

## Analysis stage 1

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
from pathlib import Path 
import copy ,gzip ,pickle ,json ,time 
import numpy as np ,pandas as pd ,matplotlib .pyplot as plt 
from sklearn .ensemble import RandomForestRegressor 
from IPython .display import display ,Markdown 
ROOT =next ((p for p in [Path .cwd (),*Path .cwd ().parents ]if (p /'data'/'processed'/'forecast_origins.parquet').exists ()),None )
if ROOT is None :raise FileNotFoundError ('\u8bf7\u4ece data3001-project \u6216\u5176\u5b50\u76ee\u5f55\u8fd0\u884c。')
CACHE =ROOT /'data'/'external'/'spatial_generalization_cache'/'region_selection_seed3001.pkl.gz'
OUT =ROOT /'output'/'generalization'/'champion_spatial'
OUT .mkdir (parents =True ,exist_ok =True )
if not CACHE .exists ():raise FileNotFoundError ('\u8bf7\u5148\u8fd0\u884c 01_region_selection_and_cache_CN.ipynb')
cache =pickle .load (gzip .open (CACHE ,'rb'));chosen =cache ['chosen'].copy ()
plt .style .use ('seaborn-v0_8-whitegrid');plt .rcParams .update ({'figure.dpi':120 ,'axes.spines.top':False ,'axes.spines.right':False })
print ('Cache settings:',cache ['settings']);display (cache ['region_selection'][cache ['region_selection'].status .str .startswith ('R')])

## Analysis stage 2

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
R=6371.; H=24; L=72
def latlon_to_local_km(olat,olon,lat,lon):
    a0=np.deg2rad(np.asarray(olat,float)); o0=np.deg2rad(np.asarray(olon,float)); a1=np.deg2rad(np.asarray(lat,float)); o1=np.deg2rad(np.asarray(lon,float)); dl=(o1-o0+np.pi)%(2*np.pi)-np.pi; da=a1-a0
    q=np.sin(da/2)**2+np.cos(a0)*np.cos(a1)*np.sin(dl/2)**2; c=2*np.arcsin(np.sqrt(np.clip(q,0,1))); b=np.arctan2(np.sin(dl)*np.cos(a1),np.cos(a0)*np.sin(a1)-np.sin(a0)*np.cos(a1)*np.cos(dl))
    return R*c*np.sin(b),R*c*np.cos(b)
def local_to_latlon(olat,olon,e,n):
    a0=np.deg2rad(np.asarray(olat,float)); o0=np.deg2rad(np.asarray(olon,float)); e=np.asarray(e,float); n=np.asarray(n,float); d=np.hypot(e,n); b=np.arctan2(e,n); ang=d/R
    lat=np.arcsin(np.sin(a0)*np.cos(ang)+np.cos(a0)*np.sin(ang)*np.cos(b)); lon=o0+np.arctan2(np.sin(b)*np.sin(ang)*np.cos(a0),np.cos(ang)-np.sin(a0)*np.sin(lat))
    return np.rad2deg(lat),(np.rad2deg(lon)+180)%360-180
def hav(lat1,lon1,lat2,lon2):
    lat1,lon1,lat2,lon2=map(lambda z:np.deg2rad(np.asarray(z,float)),[lat1,lon1,lat2,lon2]); da=lat2-lat1; dl=lon2-lon1; q=np.sin(da/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(np.clip(q,0,1)))

orig=pd.read_parquet(ROOT/'data'/'processed'/'forecast_origins.parquet').sort_values('forecast_case_id').reset_index(drop=True)
obs=pd.read_parquet(ROOT/'data'/'processed'/'region_A_observations.parquet',columns=['ID','segment_id','segment_step','latitude','longitude','ve','vn'])
n=len(orig); hist=orig[['forecast_case_id','ID','segment_id','history_start_step','origin_latitude','origin_longitude']].loc[orig.index.repeat(H)].reset_index(drop=True); hist['j']=np.tile(np.arange(H),n); hist['segment_step']=hist.history_start_step+hist.j
hist=hist.merge(obs,on=['ID','segment_id','segment_step'],validate='many_to_one').sort_values(['forecast_case_id','j']); hist['east'],hist['north']=latlon_to_local_km(hist.origin_latitude,hist.origin_longitude,hist.latitude,hist.longitude); hv=hist[['east','north','ve','vn']].to_numpy().reshape(n,H,4)
X=np.column_stack([orig.origin_latitude,orig.origin_longitude,hv.reshape(n,-1)]).astype(float)
t=orig[['forecast_case_id','ID','segment_id','target_start_step','origin_latitude','origin_longitude']].loc[orig.index.repeat(L)].reset_index(drop=True); t['lead']=np.tile(np.arange(1,L+1),n); t['segment_step']=t.target_start_step+t.lead-1
t=t.merge(obs[['ID','segment_id','segment_step','latitude','longitude']],on=['ID','segment_id','segment_step'],validate='many_to_one').sort_values(['forecast_case_id','lead']); t['east'],t['north']=latlon_to_local_km(t.origin_latitude,t.origin_longitude,t.latitude,t.longitude); Y=t[['east','north']].to_numpy().reshape(n,144).astype(float)
train=np.flatnonzero(orig.partition.eq('training')); assert X.shape==(5117,98) and len(train)==3591

ext=chosen.loc[chosen.partition.eq('test')].sort_values(['forecast_case_id','window_hour']).copy(); rows=[]
for case,g in ext.groupby('forecast_case_id',sort=False):
    g=g.sort_values('window_hour'); assert len(g)==96; o=g.iloc[23]; he=g.iloc[:24]; fu=g.iloc[24:]
    e,north=latlon_to_local_km(o.latitude,o.longitude,he.latitude,he.longitude); x=np.r_[o.latitude,o.longitude,np.column_stack([e,north,he.ve,he.vn]).ravel()]
    te,tn=latlon_to_local_km(o.latitude,o.longitude,fu.latitude,fu.longitude); y=np.column_stack([te,tn]).ravel()
    rows.append((case,str(o.ID),o.region,float(o.latitude),float(o.longitude),x,y,float(o.ve),float(o.vn)))
cases=pd.DataFrame(rows,columns=['forecast_case_id','drifter_id','region','origin_latitude','origin_longitude','X','Y','origin_ve','origin_vn'])
Xq=np.stack(cases.X); Yq=np.stack(cases.Y); print('Region-A train:',X[train].shape,'External test:',Xq.shape); display(cases.groupby('region').agg(cases=('forecast_case_id','size'),drifters=('drifter_id','nunique')))

## Analysis stage 3

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
hours=np.arange(1,73)[None,:]; last=Xq[:,2:].reshape(-1,24,4)[:,-1,2:4]
preds={}
preds['Velocity Persistence']=np.stack([last[:,0,None]*3.6*hours,last[:,1,None]*3.6*hours],axis=-1).reshape(-1,144)
train_last=hv[train,-1,2:4]; mean_v=train_last.mean(0); preds['Region-A Mean Flow']=np.tile(np.stack([mean_v[0]*3.6*hours.ravel(),mean_v[1]*3.6*hours.ravel()],axis=1).ravel(),(len(Xq),1))

# Optimised Weighted KNN: spatial_boost, K=24, p=3, direct targets.
mu=X[train].mean(0); sd=X[train].std(0); sd[sd<1e-12]=1; w=np.ones(98); w[:2]=8; A=(X[train]-mu)/sd*np.sqrt(w); B=(Xq-mu)/sd*np.sqrt(w)
d2=np.maximum((np.sum(B*B,1,keepdims=True)+np.sum(A*A,1)[None,:]-2*B@A.T)/98,0); ix=np.argpartition(d2,23,axis=1)[:,:24]; ds=np.sqrt(np.take_along_axis(d2,ix,axis=1)); ww=(ds+1e-8)**-3; ww/=ww.sum(1,keepdims=True); preds['Optimised Weighted KNN']=np.einsum('nk,nko->no',ww,Y[train][ix],optimize=True)

# Optimised Random Forest: parameters fixed by grouped CV in RQ1.
rf=RandomForestRegressor(n_estimators=600,max_depth=24,min_samples_split=20,min_samples_leaf=8,max_features=1.0,max_samples=.9,random_state=2026,n_jobs=-1)
rf.fit(X[train],Y[train]); preds['Optimised Random Forest']=rf.predict(Xq)

# Optimised FNN: exact 128-64, L2=.001, Adam, 129 epochs, three fixed seeds.
def init(s,rng): return [{'W':rng.normal(0,np.sqrt(2/a),(a,b)),'b':np.zeros(b)} for a,b in zip(s[:-1],s[1:])]
def forward(p,z):
    acts=[z]; zs=[]
    for j,q in enumerate(p): z0=z@q['W']+q['b']; zs.append(z0); z=np.maximum(z0,0) if j<len(p)-1 else z0; acts.append(z)
    return z,acts,zs
def grads(p,xb,yb):
    pr,a,z=forward(p,xb); d=2*(pr-yb)/(pr-yb).size; out=[None]*len(p)
    for j in range(len(p)-1,-1,-1): out[j]={'W':a[j].T@d+.002*p[j]['W'],'b':d.sum(0)}; d=(d@p[j]['W'].T)*(z[j-1]>0) if j else d
    return out
xm=X[train].mean(0); xs=X[train].std(0); xs[xs<1e-12]=1; ym=Y[train].mean(0); ys=Y[train].std(0); ys[ys<1e-12]=1; xt=(X[train]-xm)/xs; yt=(Y[train]-ym)/ys; xquery=(Xq-xm)/xs
members=[]
for seed in [2026,2037,2048]:
    rng=np.random.default_rng(seed); p=init((98,128,64,144),rng); m=[{'W':np.zeros_like(q['W']),'b':np.zeros_like(q['b'])} for q in p]; v=copy.deepcopy(m); step=0
    for epoch in range(129):
        perm=rng.permutation(len(xt))
        for start in range(0,len(xt),256):
            b=perm[start:start+256]; gg=grads(p,xt[b],yt[b]); step+=1
            for j in range(len(p)):
                for key in ('W','b'):
                    m[j][key]=.9*m[j][key]+.1*gg[j][key]; v[j][key]=.999*v[j][key]+.001*gg[j][key]**2; p[j][key]-=.001*(m[j][key]/(1-.9**step))/(np.sqrt(v[j][key]/(1-.999**step))+1e-8)
    members.append(forward(p,xquery)[0]*ys+ym)
preds['Optimised FNN']=np.mean(members,axis=0)
print('Models:',list(preds))

## Analysis stage 4

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
long=[]
olat=np.repeat(cases.origin_latitude.to_numpy(),72); olon=np.repeat(cases.origin_longitude.to_numpy(),72)
for model,pflat in preds.items():
    p=pflat.reshape(-1,72,2); truth=Yq.reshape(-1,72,2); plat,plon=local_to_latlon(olat,olon,p[:,:,0].ravel(),p[:,:,1].ravel()); tlat,tlon=local_to_latlon(olat,olon,truth[:,:,0].ravel(),truth[:,:,1].ravel()); err=hav(plat,plon,tlat,tlon).reshape(-1,72); ite=np.trapezoid(np.c_[np.zeros(len(err)),err],x=np.arange(73),axis=1)
    q=cases[['forecast_case_id','drifter_id','region']].copy(); q['model']=model; q['error_72h_km']=err[:,-1]; q['ite_km_h']=ite; long.append(q)
results=pd.concat(long,ignore_index=True)
region_metrics=(results.groupby(['region','model']).agg(n_cases=('forecast_case_id','size'),n_drifters=('drifter_id','nunique'),median_72h_error_km=('error_72h_km','median'),rmse_72h_km=('error_72h_km',lambda x:np.sqrt(np.mean(x*x))),median_ite_km_h=('ite_km_h','median')).reset_index())
summary=(region_metrics.groupby('model').agg(mean_region_rmse_km=('rmse_72h_km','mean'),median_region_rmse_km=('rmse_72h_km','median'),mean_region_median_error_km=('median_72h_error_km','mean'),mean_region_median_ite_km_h=('median_ite_km_h','mean'),regions=('region','nunique')).reset_index().sort_values('mean_region_rmse_km'))
base=float(summary.loc[summary.model.eq('Velocity Persistence'),'mean_region_rmse_km'].iloc[0]); summary['improvement_vs_persistence_pct']=100*(base-summary.mean_region_rmse_km)/base
display(summary.round(2)); display(region_metrics.pivot(index='model',columns='region',values='rmse_72h_km').round(1))
fig,ax=plt.subplots(figsize=(10,5)); q=summary.sort_values('mean_region_rmse_km',ascending=False); colors=['#D81B60' if x=='Optimised FNN' else '#1B9E77' if 'KNN' in x else '#377EB8' if 'Forest' in x else '#AAAAAA' for x in q.model]; ax.barh(q.model,q.mean_region_rmse_km,color=colors); [ax.text(v+1,i,f'{v:.1f}',va='center') for i,v in enumerate(q.mean_region_rmse_km)]; ax.set_xlabel('Mean of 10 region-level 72-h RMSEs (km)'); ax.set_title('Region A champions transferred without retraining',fontweight='bold'); fig.tight_layout(); fig.savefig(OUT/'champion_spatial_generalization_summary.png',dpi=200,bbox_inches='tight'); plt.show()
results.to_csv(OUT/'champion_spatial_case_metrics.csv',index=False); region_metrics.to_csv(OUT/'champion_spatial_region_metrics.csv',index=False); summary.to_csv(OUT/'champion_spatial_summary.csv',index=False)

## Analysis stage 5

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
wide=results.pivot_table(index=['region','drifter_id','forecast_case_id'],columns='model',values='error_72h_km').reset_index()
rng=np.random.default_rng(2026); boot_rows=[]
for model in [x for x in preds if x!='Velocity Persistence']:
    region_diffs=[]
    for region,g in wide.groupby('region'):
        a=g[model].to_numpy(); b=g['Velocity Persistence'].to_numpy(); point=np.sqrt(np.mean(a*a))-np.sqrt(np.mean(b*b)); draws=[]
        for _ in range(2000):
            ix=rng.integers(0,len(g),len(g)); draws.append(np.sqrt(np.mean(a[ix]**2))-np.sqrt(np.mean(b[ix]**2)))
        region_diffs.append(np.asarray(draws))
    mean_draw=np.mean(region_diffs,axis=0); point=float(summary.loc[summary.model.eq(model),'mean_region_rmse_km'].iloc[0]-base); lo,hi=np.quantile(mean_draw,[.025,.975])
    boot_rows.append({'model':model,'mean_region_rmse_difference_vs_persistence_km':point,'ci_lower_km':lo,'ci_upper_km':hi,'ci_excludes_zero':bool(lo>0 or hi<0)})
bootstrap=pd.DataFrame(boot_rows).sort_values('mean_region_rmse_difference_vs_persistence_km'); display(bootstrap.round(2)); bootstrap.to_csv(OUT/'champion_spatial_bootstrap_vs_persistence.csv',index=False)

## Analysis stage 6

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
best =summary .iloc [0 ];learned =summary [summary .model .str .startswith ('Optimised')].iloc [0 ]
display (Markdown (f'''### \u4e3b\u8981\u7ed3\u8bba

- \u5341\u4e2a\u5916\u90e8\u533a\u57df\u4e2d，\u6574\u4f53\u6700\u597d\u7684\u65b9\u6cd5\u662f **{best .model }**，mean-of-region RMSE \u4e3a **{best .mean_region_rmse_km :.1f} km**。
- \u51a0\u519b\u5b66\u4e60\u6a21\u578b\u4e2d\u6700\u597d\u7684\u662f **{learned .model }**，RMSE \u4e3a **{learned .mean_region_rmse_km :.1f} km**，\u76f8\u5bf9 persistence \u6539\u5584 **{learned .improvement_vs_persistence_pct :.1f}%**。
- \u5982\u679c Region A \u5185\u6700\u5f3a\u7684\u590d\u6742\u6a21\u578b\u5728\u5916\u90e8\u533a\u57df\u660e\u663e\u843d\u540e\u4e8e\u7b80\u5355\u6a21\u578b，\u8fd9\u4e0d\u662f“\u6a21\u578b\u5b8c\u5168\u6ca1\u7528”，\u800c\u662f\u8bf4\u660e\u7edd\u5bf9\u7ecf\u7eac\u5ea6\u53ca\u5c40\u90e8\u8f68\u8ff9\u6a21\u5f0f\u53d1\u751f\u4e86 distribution shift。
- \u8fd9\u9879\u5b9e\u9a8c\u68c0\u9a8c\u7684\u662f **zero-shot spatial transfer**：\u6ca1\u6709\u5728\u5916\u90e8\u533a\u57df\u91cd\u65b0\u8bad\u7ec3。\u5b83\u6bd4“\u5230\u6bcf\u4e2a\u65b0\u533a\u57df\u91cd\u65b0\u5212\u8bad\u7ec3\u96c6”\u66f4\u4e25\u683c，\u4e5f\u66f4\u63a5\u8fd1\u771f\u6b63\u7684\u7a7a\u95f4\u6cdb\u5316。
- \u7ed3\u679c\u53ea\u9002\u7528\u4e8e\u8fd9\u5341\u4e2a\u56fa\u5b9a\u6d77\u57df\u548c2018–2022\u65f6\u95f4\u7a97，\u4e0d\u80fd\u76f4\u63a5\u5ba3\u79f0\u5168\u7403\u5e73\u5747\u6027\u80fd。
'''))
print ('Saved to',OUT )